# Agentic Healthcare Assistant for Medical Task Automation
### Applied Generative AI Specialization — Capstone

**Author:** Jithendran Sellamuthu

A LangGraph agent that books appointments, manages records, summarizes histories, and
searches disease information — with a zero-hallucination guardrail, human-in-the-loop
writes, long-term memory, PHI-safe logging, and an LLMOps evaluation harness.

This notebook is the **narrated entry point**. The engineered code lives in the
`src/hcasst/` package; each section below imports it and runs a requirement live.

Runs **offline by default** (deterministic MockLLM, no API key needed). Set
`ANTHROPIC_API_KEY` to use real Claude.

## 0. Environment

Put `src/` and `config/` on the path so the `hcasst` package imports.

In [1]:
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent          # notebook run from a subfolder
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "config"))

from config.settings import get_settings
s = get_settings()
print("Backend:", "Claude" if s.llm_available else "Offline Mock")
print("Embeddings:", "SentenceTransformer" if s.use_st_embeddings else "Hashing")
print("Max guardrail loops:", s.max_agent_loops)

Backend: Offline Mock
Embeddings: Hashing
Max guardrail loops: 3


## 1. Data layer — typed models, seed + instructor data

Patient data loads from JSON seed **and** the instructor dataset (`records.xlsx` +
SOAP PDFs) via a thin adapter, all into the same typed `PatientHistory` / `EvidenceItem`
models. *Requirements: EHR/Patient DB, structured + unstructured records.*

In [2]:
from hcasst.tools.records import RecordsManager

records = RecordsManager()
patients = records.list_patients()
print(f"{len(patients)} patients loaded (seed + instructor):")
for p in patients:
    print("  ", p.id, "|", p.label)

7 patients loaded (seed + instructor):
   seed-001 | Raman Iyer (male, ~71y)
   seed-002 | Meera Nair (female, ~58y)
   ext-rahul-negi | Rahul Negi (Male)
   ext-rebeca-nagle | Rebeca Nagle (Female)
   ext-ramesh-kulkarni | Ramesh Kulkarni (Male, ~68y)
   ext-anjali-mehra | Anjali Mehra (Female, ~36y)
   ext-david-thompson | David Thompson (Male, ~54y)


In [3]:
# A SOAP PDF parsed into itemized, citable evidence
h = records.get_patient_history("ext-david-thompson")
print("Patient:", h.patient.label)
for i in h.items:
    print(f"  {i.evidence_id} [{str(i.kind)}] {i.text[:60]}")

Patient: David Thompson (Male, ~54y)
  E1 [EvidenceKind.CONDITION] Type 2 Diabetes Mellitus (E11.9)
  E2 [EvidenceKind.NOTE] David, a 51-year-old male, presents for follow-up of Type 2 
  E3 [EvidenceKind.OBSERVATION] Vitals: BP 140/88, Pulse 82, Temp 98.7°F. Physical exam unre
  E4 [EvidenceKind.NOTE] Increase metformin dosage to 1000mg BID. Order HbA1c, Lipid 


## 2. RAG — FAISS + SentenceTransformers

Patient evidence is embedded and retrieved by **meaning**. A query about kidney
function surfaces the CKD, eGFR, and renal-diet items even without shared keywords.
*Requirement: FAISS vector DB for patient summaries.*

In [4]:
from hcasst.rag.ingest import build_patient_index

ckd = records.get_patient_history("seed-001")
index = build_patient_index(ckd)
print("Indexed", len(index), "documents\n")
for r in index.query("declining kidney function and treatment", k=4):
    print(f"  {r.score:.3f}  {r.document.citation}  {r.document.text[:55]}")

Indexed 8 documents

  0.158  [E1]  Chronic kidney disease, stage 3b (2024-11-10)
  0.000  [E4]  Lisinopril 10 mg daily (2024-11-10)
  0.000  [E3]  Hypertension (2018-07-15)
  0.000  [E2]  Type 2 diabetes mellitus (2019-03-22)


## 3. Long-term memory — durable + semantic recall

Memory stores interaction context that is **not** in the medical record
(preferences, caregiver info), persisted in SQLite and recalled semantically.
*Requirement: memory module for long-term patient context.*

In [5]:
from hcasst.memory.store import MemoryManager

mem = MemoryManager()
mem.remember("seed-001", "Patient prefers morning appointments; anxious about dialysis.", kind="preference")
mem.remember("seed-001", "Daughter Priya is the primary caregiver and attends visits.", kind="caregiver")
mem.index_patient(ckd)
print("Recall — 'who is the family caregiver':")
print(mem.recall("seed-001", "who is the family caregiver", k=2).as_evidence())

Recall — 'who is the family caregiver':
[M2] Daughter Priya is the primary caregiver and attends visits.
[E2] Type 2 diabetes mellitus (2019-03-22)


## 4. Planner — multi-step decomposition

The planner decomposes a multi-intent query into an ordered, **validated** tool plan
(invalid/hallucinated tools are dropped; patient lookup is always first).
*Requirement: planner decomposes query and selects tools.*

In [6]:
from hcasst.agent.planner import make_plan, specialty_of, urgency_of

q = ("My 70-year-old father has chronic kidney disease. I want to book a nephrologist "
     "for him. Also, can you summarize the latest treatment methods?")
for step in make_plan(q, patient_hint="seed-001"):
    print(" ", step["tool"], "—", step["goal"])
print("specialty:", specialty_of(q), "| urgency:", urgency_of("need this urgently"))

  get_patient_history — Identify patient and retrieve history
  book_appointment — Find and propose appointment slots
  summarize_history — Summarize medical history with citations
  search_disease_info — Search trusted medical sources
specialty: nephrology | urgency: urgent


## 5. Zero-hallucination guardrail

Generated answers must cite evidence. Uncited answers are regenerated up to 3×
("loop break of 3"), then caveated rather than emitted as fact.
*Requirement: zero-hallucination / grounding.*

In [7]:
from hcasst.agent.guardrails import generate_grounded, citation_presence

print("cited? ", citation_presence("CKD stage 3 [E1]"))
print("uncited?", citation_presence("CKD with no citation"))

ans, verdict, loops = generate_grounded(
    lambda: "Patient has CKD stage 3 [E1] on lisinopril [E4].", "E1: CKD\nE4: lisinopril")
print(f"\ngrounded={verdict['grounded']} in {loops} loop(s)")

ans2, v2, l2 = generate_grounded(lambda: "Unsupported claim, no citation.", "E1: CKD")
print(f"ungrounded -> caveat appended: {'[unverified]' in ans2} after {l2} loops")

2026-06-22 06:19:44,708 | WARNING | hcasst.guardrails | Attempt 1 not grounded: ['answer contains no citations']


2026-06-22 06:19:44,710 | WARNING | hcasst.guardrails | Attempt 2 not grounded: ['answer contains no citations']


2026-06-22 06:19:44,710 | WARNING | hcasst.guardrails | Attempt 3 not grounded: ['answer contains no citations']


cited?  True
uncited? False

grounded=True in 1 loop(s)
ungrounded -> caveat appended: True after 3 loops


## 6. PHI-safe logging

Identifiers (DOB, phone, email, SSN) are masked before any trace is persisted.
Birth **year** is kept so age stays computable. *Requirement: PHI handling.*

In [8]:
from hcasst.obs.logging import mask_phi

raw = "David Thompson DOB: 03/14/1972 Phone: +91-98450-11223 email d@x.com SSN 123-45-6789"
print("RAW :", raw)
print("MASK:", mask_phi(raw))

RAW : David Thompson DOB: 03/14/1972 Phone: +91-98450-11223 email d@x.com SSN 123-45-6789
MASK: David Thompson DOB: **/**/1972 Phone: <phone> email <email> SSN <ssn>


## 7. The agent — golden CKD scenario, end to end

The full LangGraph agent runs the sample scenario: plan → identify → book (HITL) →
summarize → search → guardrail → respond → memory. The booking is **proposed**, not
auto-executed. *Requirement: agent execution flow, HITL.*

In [9]:
from hcasst.agent.graph import HealthAssistant, serialize_state

agent = HealthAssistant()
state = agent.run(q, patient_hint="seed-001")
out = serialize_state(state)

print("PLAN:", [st["tool"] for st in out["plan"]])
print("BOOKING (needs approval):", out["booking"]["needs_approval"] if out["booking"] else None,
      "->", out["booking"]["specialty"] if out["booking"] else None)
print("\nRESPONSE:\n", state["response"][:600])

2026-06-22 06:19:45,289 | INFO    | hcasst.graph | Running session 4c7b88bb374d: My 70-year-old father has chronic kidney disease. I want to book a nephrologist for him. Also, can you summarize the latest treatment methods?


PLAN: ['get_patient_history', 'book_appointment', 'summarize_history', 'search_disease_info']
BOOKING (needs approval): True -> nephrology

RESPONSE:
 **Patient:** Raman Iyer (male, ~71y)

### Medical History Summary
Summary (offline mode): statements below are drawn from the provided evidence [E1]. Connect an ANTHROPIC_API_KEY for a clinically richer summary.

### Proposed Appointment — awaiting your approval
- **Specialty:** nephrology
- **Provider:** Dr. Aisha Rahman, MD (Nephrology)
- **When:** 2026-06-01T09:14:00Z

> No booking is made until you approve it (human-in-the-loop).

### Latest Information (cited)
Disease information (offline mode): see the cited source snippets below for current guidance [OfflineKB]. Connect an ANTHROPIC_API


In [10]:
# Per-node traces (PHI-masked) — the observability layer
from hcasst.db.store import Store
for t in Store().traces(state["session_id"], limit=10):
    print(f"  {t['step']:10} | {t['name']:22} | {t['status']}")

  plan       | planner                | ok
  tool       | get_patient_history    | ok
  guardrail  | human_approval_gate    | blocked
  tool       | summarize_history      | ok
  tool       | search_disease_info    | ok
  guardrail  | grounding_check        | ok
  respond    | assemble               | ok


## 8. Human-in-the-loop write — approve the booking

The proposal commits only on explicit confirmation. The DB row id is the single
source of truth, so a fresh scheduler rehydrates it from disk.

In [11]:
prop = state["booking_proposal"]
print("before:", prop.status)
confirmed = agent.scheduler.confirm_booking(prop.proposal_id)
print("after :", confirmed.status, "(scheduled = human approved)")

before: proposed
after : scheduled (scheduled = human approved)


## 9. Records — add / update (HITL, audited)

Attendants add unstructured notes or structured items, or update existing evidence;
every change is audited. *Requirement: manage records, structured + unstructured.*

In [12]:
from hcasst.models import EvidenceKind

note = records.propose_visit("seed-001", "Follow-up: eGFR stable, continue regimen.", kind=EvidenceKind.NOTE)
records.confirm_visit(1)
records.propose_visit("seed-001", "Chronic kidney disease, stage 4", target_evidence_id="E1")
records.confirm_visit(2)
e1 = next(i for i in records.get_patient_history("seed-001").items if i.evidence_id == "E1")
print("audit rows:", len(records.store.record_audit("seed-001")))
print("E1 now:", e1.text)

audit rows: 2
E1 now: Chronic kidney disease, stage 4


## 10. Disease search — trusted external sources

MedlinePlus (NLM) + PubMed (NCBI) live, with an offline curated KB fallback.
*Requirement: disease info from trusted sources.*

In [13]:
from hcasst.tools.disease_search import DiseaseSearch

results = DiseaseSearch().search("chronic kidney disease treatment")
for r in results[:3]:
    print(f"  {r.citation}  {r.title[:60]}")

  [OfflineKB]  Chronic Kidney Disease


## 11. Evaluation — LLMOps harness

Golden cases scored on tool-selection F1, booking success, citation coverage, QA
correctness, and grounding; persisted to `eval_runs`. Offline, QA/citation are low
by design (mock text); tool/booking/grounding are valid.
*Requirement: QAEval + per-module metrics.*

In [14]:
from hcasst.eval.runner import run_evaluation

report = run_evaluation()
print("run", report["run_id"], "| llm:", report["llm"])
for k, v in report["metrics"].items():
    print(f"  {k:<26} {v:>6.3f}")

2026-06-22 06:19:45,406 | INFO    | hcasst.eval | Eval run b51f07eaf810 (4 cases, llm=mock)


2026-06-22 06:19:45,408 | INFO    | hcasst.graph | Running session 7743ca42f8a6: My 70-year-old father has chronic kidney disease. I want to book a nephrologist for him. Also, can you summarize the latest treatment methods?


2026-06-22 06:19:45,522 | INFO    | hcasst.graph | Running session 47cfad5b2c63: Book a cardiologist for Meera and summarize her heart history.


2026-06-22 06:19:45,580 | INFO    | hcasst.graph | Running session a24932955786: Summarize my father's medical history.


2026-06-22 06:19:45,620 | INFO    | hcasst.graph | Running session 182fa8e2cfb5: What are the latest treatment methods for chronic kidney disease?


run b51f07eaf810 | llm: mock
  booking.success             1.000
  grounding.grounded          1.000
  planning.tool_f1            0.950
  search.citation_coverage    0.500
  search.qa_correctness       0.000
  summary.citation_coverage   0.500
  summary.qa_correctness      0.250
  OVERALL                     0.600


## 12. Summary

All functional requirements met; four quality bars enforced (zero-hallucination,
HITL, PHI-safe, reproducible). Runs offline deterministically and upgrades to real
Claude with a key. The full dashboard (`streamlit run app/streamlit_app.py`) exposes
Assistant, Patient, Doctor, Evaluation, Traces, and Documentation views.

**Architecture principle:** program to the interface, not the implementation — which
is why mock↔Claude, hashing↔SentenceTransformer, and seed↔instructor-data all swap
without touching callers.